# Conectándose a Autodesk Fusion (APS)

Este notebook muestra cómo usar el paquete `fusion_connect` para conectarse a la API de **Autodesk Platform Services (APS)** que usa Autodesk Fusion, listar los *hubs* y *proyectos* visibles para tu aplicación, y descargar la última versión de un archivo.

## Requisitos

1. Crear una aplicación en https://aps.autodesk.com/ y obtener su `Client ID` y `Client Secret`.
2. Provisionar la aplicación en el hub de Fusion Team (o BIM 360 / ACC) para que pueda leer los datos del hub.
3. Instalar el paquete:

   ```bash
   cd packages/fusion_connect
   pip install .
   ```
4. Exportar las credenciales como variables de entorno:

   ```bash
   export APS_CLIENT_ID="..."
   export APS_CLIENT_SECRET="..."
   ```

In [ ]:
from fusion_connect import APSAuth, FusionClient

auth = APSAuth.from_env()
fusion = FusionClient(auth)

## 1. Listar hubs y proyectos

Un *hub* representa una cuenta de Fusion Team (o BIM 360 / ACC). Cada hub contiene uno o más *proyectos*.

In [ ]:
hubs = fusion.list_hubs()
for hub in hubs:
    print(hub['id'], '-', hub['attributes']['name'])
    for project in fusion.list_projects(hub['id']):
        print('   ', project['id'], '-', project['attributes']['name'])

## 2. Explorar carpetas

Reemplaza `hub_id` y `project_id` con los identificadores impresos arriba.

In [ ]:
hub_id = 'b.xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'
project_id = 'a.yyyyyyyy'

top_folders = fusion.get_top_folders(hub_id, project_id)
for folder in top_folders:
    print(folder['id'], '-', folder['attributes']['displayName'])

In [ ]:
folder_id = top_folders[0]['id']
for entry in fusion.list_folder_contents(project_id, folder_id):
    print(entry['type'], entry['id'], entry['attributes'].get('displayName'))

## 3. Descargar la última versión de un item

Cada *item* en Fusion (por ejemplo un diseño `.f3d`) puede tener múltiples versiones. La versión "tip" es la más reciente.

In [ ]:
item_id = 'urn:adsk.wipprod:dm.lineage:xxxxxxxx'

tip = fusion.get_tip_version(project_id, item_id)
version_id = tip['data']['id']
print('Tip version:', version_id)

fusion.download_version(project_id, version_id, '/tmp/fusion_part.bin')